# 📅 Day 05: Monotonic Structures — Deque & Stack for Sliding Windows

---

## From Monotonic Stack to Monotonic Deque

Monotonic **stack** (Chapter 03) gives you the next greater/smaller element. Monotonic **deque** adds the ability to efficiently get the max/min of a **sliding window** in O(1) per element.

### The Core Idea

Maintain a deque where elements are always in decreasing order (for max) or increasing order (for min). When the window slides:
- **Back:** pop smaller elements before adding new one (they'll never be the max)
- **Front:** pop if it's outside the window
- **Max/Min:** always the front of the deque

Think of it like a VIP line at a club. When a taller person arrives, everyone shorter in front of them will NEVER be the tallest. So kick them out. The tallest person is always at the front.

```
╔═══════════════════════════════════════════════════════════════════════╗
║  MONOTONIC DEQUE = Sliding window max/min in O(n) total            ║
║                                                                     ║
║  Max deque: decreasing order → front = max                         ║
║  Min deque: increasing order → front = min                         ║
║  Store INDICES (not values) to check window bounds                 ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 🎯 Problem 1: Sliding Window Maximum (LC #239)

Return the max of each window of size k.

### Step-by-Step Trace

```
  nums = [1, 3, -1, -3, 5, 3, 6, 7], k = 3
  Deque stores INDICES (shown as values for clarity)
  
  i=0: num=1.  Deque: [1].          Window not full yet.
  i=1: num=3.  Pop 1 (3>1). Deque: [3].  Window not full yet.
  i=2: num=-1. Deque: [3, -1].      Window [1,3,-1] → max=3 ✓
  i=3: num=-3. Deque: [3, -1, -3].  Window [3,-1,-3] → max=3 ✓
  i=4: num=5.  Pop all (5>3). Deque: [5].  Window [-1,-3,5] → max=5 ✓
  i=5: num=3.  Deque: [5, 3].       Window [-3,5,3] → max=5 ✓
  i=6: num=6.  Pop 3, pop 5 (6>5). Deque: [6].  Window [5,3,6] → max=6 ✓
  i=7: num=7.  Pop 6 (7>6). Deque: [7].  Window [3,6,7] → max=7 ✓
  
  Result: [3, 3, 5, 5, 6, 7]
```

In [ ]:
from collections import deque

def maxSlidingWindow(nums, k):
    """
    Monotonic decreasing deque of INDICES.
    Each element enters and leaves the deque exactly once → O(n) total.
    """
    dq = deque()  # Stores indices, values are decreasing
    result = []
    
    for i, num in enumerate(nums):
        # Remove elements outside window from front
        if dq and dq[0] < i - k + 1:
            dq.popleft()
        
        # Remove smaller elements from back (they'll never be max)
        while dq and nums[dq[-1]] < num:
            dq.pop()
        
        dq.append(i)
        
        # Window is full, record the max (always at front)
        if i >= k - 1:
            result.append(nums[dq[0]])
    
    return result

print(maxSlidingWindow([1,3,-1,-3,5,3,6,7], 3))  # [3,3,5,5,6,7]

---

## 🎯 Problem 2: Shortest Subarray with Sum at Least K (LC #862)

Find the shortest subarray with sum ≥ k. **Can have negative numbers!**

### Why This is Harder Than It Looks

With all positives, a simple sliding window works. But **negatives break the sliding window** — shrinking from the left might INCREASE the sum (if you remove a negative). So the simple two-pointer approach fails.

**Solution:** Use **prefix sums + monotonic deque**.
- For each `j`, we need the smallest `i < j` where `prefix[j] - prefix[i] >= k`
- The deque maintains indices with increasing prefix sums
- Pop from front when condition met (found a valid subarray)
- Pop from back when current prefix ≤ back (current is a better starting point)

In [ ]:
def shortestSubarray(nums, k):
    """
    Prefix sum + monotonic deque.
    Time: O(n), Space: O(n)
    """
    n = len(nums)
    prefix = [0] * (n + 1)
    for i in range(n):
        prefix[i+1] = prefix[i] + nums[i]
    
    dq = deque()  # Indices with increasing prefix sums
    result = float('inf')
    
    for j in range(n + 1):
        # Try to find valid subarray (pop from front)
        while dq and prefix[j] - prefix[dq[0]] >= k:
            result = min(result, j - dq.popleft())
        
        # Maintain increasing order (pop from back)
        while dq and prefix[j] <= prefix[dq[-1]]:
            dq.pop()  # Current j is a better starting point
        
        dq.append(j)
    
    return result if result != float('inf') else -1

print(shortestSubarray([2,-1,2], 3))  # 3
print(shortestSubarray([84,-37,32,40,95], 167))  # 3

---

## 📊 Pattern Map

```
╔═══════════════════════════════════════════════════════════════════════╗
║  MONOTONIC STRUCTURE PATTERNS                                      ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                     ║
║  Sliding window max/min    → Monotonic deque                       ║
║  Next greater/smaller      → Monotonic stack (Chapter 03)          ║
║  Max subarray with neg     → Prefix sum + monotonic deque          ║
║  Min/max of all subarrays  → Two deques (min + max)               ║
║                                                                     ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 239 | Sliding Window Maximum | Hard | Monotonic decreasing deque |
| 862 | Shortest Subarray with Sum ≥ K | Hard | Prefix sum + deque |
| 1438 | Longest Subarray Abs Diff ≤ Limit | Medium | Two deques (min + max) |
| 1696 | Jump Game VI | Medium | Deque for max in window |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Monotonic deque: O(n) total for sliding window max/min        │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Store INDICES — you need them to check window bounds          │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Negatives break sliding window → use prefix sum + deque       │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Each element enters/leaves deque once → amortized O(1) each   │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Custom Structures